In [ ]:
# ============================================================
# ENSEMBLE MODEL TRAINING FOR TWO SCENARIOS
# Scenario I  : Original Dataset
# Scenario II : PDA + TM Automata-Enhanced Dataset
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier
)

from sklearn.svm import SVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

# ============================================================
# BASIC SETTINGS
# ============================================================

TARGET_COL = "Recurred"
RANDOM_STATE = 42
OUT_DIR = "ensemble_results"

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# TARGET ENCODING
# ============================================================

def encode_target_column(df, target_col):
    df = df.copy()

    if df[target_col].dtype == "object":
        le = LabelEncoder()
        df[target_col] = le.fit_transform(df[target_col].astype(str))
        print("Target classes:", list(le.classes_))

    return df


scenario_1_df = encode_target_column(scenario_1_df, TARGET_COL)
scenario_2_df = encode_target_column(scenario_2_df, TARGET_COL)

In [ ]:
# ============================================================
# PREPROCESSING PIPELINE
# ============================================================

def get_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [
        col for col in X.columns
        if col not in numeric_cols
    ]

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", get_onehot_encoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols)
        ],
        remainder="drop"
    )

    return preprocessor